# 13 — Physical-World Attacks & Universal Perturbations — Lab

This lab exercises Expectation Over Transformation (EOT) based patch optimization with Non-Printable Score (NPS) printability constraints. You will implement the complete EOT patch optimization loop, measure fooling rate across held-out test images, and compute the printability of the final patch. By the end, you will have generated a physical adversarial patch and validated its robustness to real-world transformations.

## Objectives

- Explain how Expectation Over Transformation (EOT) makes a perturbation robust to rotation, scale, and lighting changes by optimizing over a distribution of transformations.
- Generate a printable adversarial patch using EOT optimization with Non-Printable Score (NPS) printability loss.
- Measure the fooling rate of an adversarial patch across a diverse image set.

## Prerequisites

You should have completed the lecture for m13 (Physical-World Attacks & Universal Perturbations). You should understand Expectation Over Transformation (EOT), adversarial patch design, Non-Printable Score (NPS) loss, and the concept of fooling rate. Prior experience with white-box adversarial attacks (m08, m09) and classifier gradient computation is helpful but not required.

## Required Software/Packages

- PyTorch (version 2.0+) and torchvision
- NumPy and Matplotlib
- torchvision.datasets for CIFAR-10 or ImageNet-subset
- torchvision.models for pre-trained ResNet-50

All packages are available in the standard environment. No additional pip installs are required.

## Environment Setup

Run the cell below to import all required libraries and set up the environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torchvision.transforms import functional as TF
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Background

In the lecture, you learned that digital adversarial perturbations fail in the physical world because rotation, scale, and lighting transform the image. Expectation Over Transformation (EOT) addresses this by optimizing a perturbation such that the loss is minimized in expectation over a distribution of transformations. In this lab, you will apply EOT to a small adversarial patch, constrain it to be printable using Non-Printable Score (NPS) loss, and measure how well the patch fools a pre-trained image classifier across diverse viewing conditions.

## Exercise Instructions

1. Load a pre-trained ResNet-50 model and CIFAR-10 test set; initialize patch parameters (32x32 pixels) at a random location in the image (3 min).
2. Implement the Expectation Over Transformation (EOT) function that samples random rotations [-15,15] degrees, scale [0.8,1.2], and brightness jitter [0.9,1.1] for each gradient step (4 min).
3. Implement Non-Printable Score (NPS) loss as L2 distance between patch pixels and clipped safe range [50,200] per channel (2 min).
4. Run 100 iterations of EOT patch optimization using Adam optimizer (lr=0.01), combining adversarial loss and NPS loss (lambda=0.1), and clipping patch pixels to [50,200] after each step (12 min).
5. Evaluate the optimized patch by overlaying it on 100 held-out test images, computing fooling rate as fraction of misclassifications (5 min).
6. Compute the NPS printability score for the final patch and display the patch and 3 example adversarial images with fooling rate and NPS score (4 min).

## Starter Code

Run the cell below to set up the model and dataset.

In [ ]:
# Load pre-trained ResNet-50 model and set to evaluation mode
model = models.resnet50(pretrained=True).to(device)
model.eval()

# Freeze all parameters (we only optimize the patch, not the model)
for param in model.parameters():
    param.requires_grad = False

# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    # Note: CIFAR-10 images are 32x32; resize to 224x224 for ResNet-50
    transforms.Resize((224, 224))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Select subset for faster training (first 10 training images for optimization, first 100 test images for evaluation)
train_images = torch.stack([train_dataset[i][0] for i in range(10)])
test_images = torch.stack([test_dataset[i][0] for i in range(100)])
test_labels = torch.tensor([test_dataset[i][1] for i in range(100)])

# Normalization for ImageNet-pretrained model
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

# Adversarial patch initialization
patch_size = 32
patch = torch.rand(1, 3, patch_size, patch_size) * 0.5 + 0.25  # Initialize in [0.25, 0.75]
patch = patch.to(device)
patch.requires_grad = True

# Random patch location (will vary per iteration)
patch_top = 0
patch_left = 0

print(f"Model: ResNet-50 (frozen)")
print(f"Train images shape: {train_images.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Patch shape: {patch.shape}")
print(f"Device: {device}")

## Your Implementation

Complete the code cells below to implement EOT patch optimization.

In [ ]:
# Step 1: Implement EOT function
def apply_eot_transformations(image, num_samples=10):
    """
    Apply random Expectation Over Transformation (EOT) transformations to an image.
    Returns a batch of transformed images.
    
    Args:
        image: Tensor of shape (1, 3, 224, 224)
        num_samples: Number of transformations to sample
    
    Returns:
        Tensor of shape (num_samples, 3, 224, 224) with transformed images
    """
    transformed = []
    
    for _ in range(num_samples):
        # Sample random rotation: [-15, 15] degrees
        angle = np.random.uniform(-15, 15)
        
        # Sample random scale: [0.8, 1.2]
        scale = np.random.uniform(0.8, 1.2)
        
        # Sample random brightness jitter: [0.9, 1.1]
        brightness = np.random.uniform(0.9, 1.1)
        
        # Apply rotation using TF.rotate (differentiable)
        img = TF.rotate(image, angle, fill=0.5)
        
        # Apply scale (resize)
        h, w = img.shape[-2:]
        new_h, new_w = int(h * scale), int(w * scale)
        img = F.interpolate(img, size=(new_h, new_w), mode='bilinear', align_corners=False)
        
        # Pad back to 224x224 if needed
        if new_h < 224 or new_w < 224:
            pad_h = max(0, (224 - new_h) // 2)
            pad_w = max(0, (224 - new_w) // 2)
            img = F.pad(img, (pad_w, 224 - new_w - pad_w, pad_h, 224 - new_h - pad_h), value=0.5)
        elif new_h > 224 or new_w > 224:
            img = img[:, :, :224, :224]
        
        # Apply brightness jitter
        img = img * brightness
        img = torch.clamp(img, 0, 1)
        
        transformed.append(img)
    
    return torch.cat(transformed, dim=0)

print("EOT function implemented.")

In [ ]:
# Step 2: Implement NPS (Non-Printable Score) loss
def nps_loss(patch, safe_range=(50/255, 200/255)):
    """
    Compute Non-Printable Score (NPS) loss.
    Measures how far patch colors deviate from a safe printable range.
    
    Args:
        patch: Adversarial patch tensor of shape (1, 3, patch_size, patch_size)
        safe_range: Tuple (min_val, max_val) representing the safe color range
    
    Returns:
        Scalar loss value
    """
    lower, upper = safe_range
    
    # Compute distance to the safe range
    too_low = torch.clamp(lower - patch, min=0)
    too_high = torch.clamp(patch - upper, min=0)
    
    # NPS loss is the L2 distance
    loss = torch.mean(too_low ** 2 + too_high ** 2)
    
    return loss

print("NPS loss function implemented.")

In [ ]:
# Step 3: Implement patch application with random placement
def apply_patch(image, patch, random_location=True):
    """
    Apply adversarial patch to an image at a random or fixed location.
    
    Args:
        image: Input image tensor of shape (1, 3, 224, 224)
        patch: Adversarial patch of shape (1, 3, patch_size, patch_size)
        random_location: If True, place patch at random location each time
    
    Returns:
        Perturbed image of shape (1, 3, 224, 224)
    """
    perturbed = image.clone()
    patch_size = patch.shape[-1]
    
    if random_location:
        # Random patch location
        top = np.random.randint(0, 224 - patch_size + 1)
        left = np.random.randint(0, 224 - patch_size + 1)
    else:
        top, left = 0, 0
    
    # Apply patch (overwrite the region)
    perturbed[:, :, top:top+patch_size, left:left+patch_size] = patch
    
    return perturbed, top, left

print("Patch application function implemented.")

### Expected Output (Step 1–3)

You should see three print statements confirming that EOT, NPS, and patch application functions are implemented. No numerical output is expected yet.

In [ ]:
# Step 4: Run 100 iterations of EOT patch optimization
optimizer = torch.optim.Adam([patch], lr=0.01)
nps_weight = 0.1  # Weight for printability loss
num_iterations = 100
num_eot_samples = 10
loss_history = []

for iteration in range(num_iterations):
    # Select a random training image
    idx = np.random.randint(0, len(train_images))
    image = train_images[idx:idx+1].to(device)
    
    # Apply patch at random location
    perturbed, top, left = apply_patch(image, patch, random_location=True)
    
    # Apply EOT transformations
    eot_images = apply_eot_transformations(perturbed, num_samples=num_eot_samples)
    
    # Normalize for ImageNet model
    eot_images_normalized = (eot_images - imagenet_mean) / imagenet_std
    
    # Compute adversarial loss (cross-entropy with random target label)
    with torch.enable_grad():
        logits = model(eot_images_normalized)
        true_label = train_dataset[idx][1]
        target_label = (true_label + 1) % 10  # Random wrong label
        adv_loss = F.cross_entropy(logits, torch.full((num_eot_samples,), target_label).to(device))
    
    # Compute NPS loss
    nps_l = nps_loss(patch, safe_range=(50/255, 200/255))
    
    # Total loss
    total_loss = adv_loss + nps_weight * nps_l
    loss_history.append(total_loss.item())
    
    # Backward and optimize
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    # Clip patch to safe printable range
    with torch.no_grad():
        patch.data = torch.clamp(patch.data, 50/255, 200/255)
    
    if (iteration + 1) % 25 == 0:
        print(f"Iteration {iteration + 1}: Loss = {total_loss.item():.4f}, Adv Loss = {adv_loss.item():.4f}, NPS Loss = {nps_l.item():.4f}")

print(f"\nOptimization complete. Final loss: {loss_history[-1]:.4f}")

### Expected Output (Step 4)

You should see loss values printed at iterations 25, 50, 75, and 100. The adversarial loss should generally decrease, indicating the patch is learning to fool the classifier. NPS loss should remain small (close to zero if patch stays in the safe range). Final loss should be lower than the initial loss.

In [ ]:
# Step 5: Evaluate the optimized patch on 100 held-out test images
fooling_count = 0
evaluation_results = []

with torch.no_grad():
    for i in range(len(test_images)):
        test_image = test_images[i:i+1].to(device)
        true_label = test_labels[i].item()
        
        # Apply patch at a random location
        perturbed, top, left = apply_patch(test_image, patch, random_location=True)
        
        # Apply a single EOT transformation (random rotation, scale, brightness)
        eot_perturbed = apply_eot_transformations(perturbed, num_samples=1)
        
        # Normalize and classify
        eot_normalized = (eot_perturbed - imagenet_mean) / imagenet_std
        logits = model(eot_normalized)
        predicted_label = logits.argmax(dim=1).item()
        
        # Check if misclassified (fooled)
        if predicted_label != true_label:
            fooling_count += 1
            evaluation_results.append((i, predicted_label, True))
        else:
            evaluation_results.append((i, predicted_label, False))

fooling_rate = fooling_count / len(test_images) * 100
print(f"\nEvaluation on 100 test images:")
print(f"Fooling count: {fooling_count} / {len(test_images)}")
print(f"Fooling rate: {fooling_rate:.2f}%")

### Expected Output (Step 5)

You should see fooling count (e.g., "Fooling count: 42 / 100") and fooling rate (e.g., "Fooling rate: 42.00%"). The exact number depends on the patch optimization quality, but expect a fooling rate in the range of 30–70 percent for a simple CIFAR-10 attack.

In [ ]:
# Step 6: Compute NPS printability score and display results
final_nps = nps_loss(patch, safe_range=(50/255, 200/255)).item()
patch_np = patch.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()
patch_np = np.clip(patch_np, 0, 1)

print(f"\nFinal Patch Statistics:")
print(f"NPS Printability Loss: {final_nps:.6f}")
print(f"Patch min pixel value: {patch_np.min():.4f} (normalized)")
print(f"Patch max pixel value: {patch_np.max():.4f} (normalized)")
print(f"Safe range: [0.196, 0.784] (i.e., [50/255, 200/255])")

# Display patch and 3 example adversarial images
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

# Subplot 1: Adversarial patch
axes[0, 0].imshow(patch_np)
axes[0, 0].set_title(f"Adversarial Patch (32x32)\nNPS Loss: {final_nps:.6f}")
axes[0, 0].axis('off')

# Subplots 2–4: Example adversarial images (fooled)
fooled_indices = [idx for idx, _, fooled in evaluation_results if fooled][:3]
for subplot_idx, img_idx in enumerate(fooled_indices, start=1):
    test_image = test_images[img_idx:img_idx+1].to(device)
    perturbed, top, left = apply_patch(test_image, patch, random_location=False)
    perturbed_np = perturbed.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()
    perturbed_np = np.clip(perturbed_np, 0, 1)
    
    row = subplot_idx // 2
    col = subplot_idx % 2
    axes[row, col].imshow(perturbed_np)
    true_label = test_labels[img_idx].item()
    axes[row, col].set_title(f"Example {subplot_idx} (Fooled)\nTrue Label: {true_label}")
    axes[row, col].axis('off')

plt.suptitle(f"EOT Adversarial Patch Optimization\nFooling Rate: {fooling_rate:.2f}% ({fooling_count}/{len(test_images)})", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nVisualization complete.")

### Expected Output (Step 6)

You should see:
1. Final patch statistics (NPS loss, min/max pixel values, safe range).
2. A 2x2 figure showing the adversarial patch (top-left) and 3 example adversarial images (other subplots).
3. The overall figure title shows the fooling rate and count.

## Comprehension Questions

1. Why does optimizing over a distribution of transformations (EOT) make the patch more robust to real-world viewing conditions than optimizing a single image? Explain how sampling 10 transformations per gradient step improves generalization to unseen rotations and scales.

2. The Non-Printable Score (NPS) loss constrains patch pixels to the range [50/255, 200/255]. Why is this range chosen, and what happens if you remove the NPS loss term (set lambda=0)? Would the fooling rate increase or decrease? Why?

## Verification

Run the cell below to verify that your implementation is correct.

In [ ]:
# Verification: Check that key outputs exist and have correct properties
verification_passed = True
messages = []

# Check 1: Patch exists and is on device
if patch is None or patch.device != device:
    verification_passed = False
    messages.append("FAIL: Patch does not exist or is not on the correct device.")
else:
    messages.append("PASS: Patch exists and is on the correct device.")

# Check 2: Loss history has 100 entries (one per iteration)
if len(loss_history) != 100:
    verification_passed = False
    messages.append(f"FAIL: Loss history has {len(loss_history)} entries, expected 100.")
else:
    messages.append("PASS: Loss history has 100 entries (one per iteration).")

# Check 3: Fooling rate is between 0 and 100
if fooling_rate < 0 or fooling_rate > 100:
    verification_passed = False
    messages.append(f"FAIL: Fooling rate {fooling_rate:.2f}% is out of range [0, 100].")
else:
    messages.append(f"PASS: Fooling rate {fooling_rate:.2f}% is valid.")

# Check 4: NPS loss is non-negative
if final_nps < 0:
    verification_passed = False
    messages.append(f"FAIL: NPS loss {final_nps:.6f} is negative.")
else:
    messages.append(f"PASS: NPS loss {final_nps:.6f} is non-negative.")

# Check 5: Patch pixels are mostly in safe range [50/255, 200/255]
safe_lower = 50/255
safe_upper = 200/255
in_range = torch.sum((patch >= safe_lower) & (patch <= safe_upper)).item()
total = patch.numel()
in_range_pct = in_range / total * 100
if in_range_pct < 80:
    verification_passed = False
    messages.append(f"FAIL: Only {in_range_pct:.1f}% of patch pixels are in safe range [50/255, 200/255].")
else:
    messages.append(f"PASS: {in_range_pct:.1f}% of patch pixels are in safe range.")

# Final result
print("\n".join(messages))
if verification_passed:
    print("\n=== ALL VERIFICATION CHECKS PASSED ===")
else:
    print("\n=== SOME VERIFICATION CHECKS FAILED ===")

## Optional Challenge

1. Extend the lab to compute UAP (Universal Adversarial Perturbation) using the DeepFool algorithm. Accumulate perturbations from multiple training images and clip to an L-infinity ball of epsilon=8/255. Compare the fooling rate of UAP vs. the patch.

2. Experiment with different values of the NPS weight (lambda). Plot fooling rate vs. lambda and observe the trade-off between attack success and printability.